In [17]:
"""Market Risk VaR Engine (single file). Run: python var_engine.py
Needs: pip install numpy pandas matplotlib"""
import os, math
from statistics import NormalDist
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ===== data =====
ASSETS = ["US_Equity", "EU_Equity", "EM_Equity", "Gov_Bonds", "Corp_Bonds", "Gold", "EURUSD"]
ANN_RET = np.array([0.08, 0.07, 0.09, 0.03, 0.04, 0.04, 0.00])
ANN_VOL = np.array([0.18, 0.20, 0.25, 0.06, 0.08, 0.15, 0.08])
CORR = np.array([
    [1.00, 0.85, 0.70, -0.20, 0.35, 0.05, 0.25],
    [0.85, 1.00, 0.75, -0.15, 0.35, 0.05, 0.35],
    [0.70, 0.75, 1.00, -0.10, 0.30, 0.10, 0.30],
    [-0.20, -0.15, -0.10, 1.00, 0.55, 0.20, -0.05],
    [0.35, 0.35, 0.30, 0.55, 1.00, 0.15, 0.05],
    [0.05, 0.05, 0.10, 0.20, 0.15, 1.00, 0.25],
    [0.25, 0.35, 0.30, -0.05, 0.05, 0.25, 1.00],
])

# Portfolio weights (sum to 1) and notional
WEIGHTS = pd.Series([0.25, 0.15, 0.10, 0.20, 0.15, 0.10, 0.05], index=ASSETS)
PORTFOLIO_VALUE = 10_000_000


def simulate_returns(n_days=2000, seed=42, df=5):
    """Correlated fat-tailed (Student-t) daily returns with a volatility-stress regime."""
    rng = np.random.default_rng(seed)
    cov = np.outer(ANN_VOL, ANN_VOL) * CORR / 252
    L = np.linalg.cholesky(cov)
    z = rng.standard_normal((n_days, len(ASSETS)))
    chi = rng.chisquare(df, size=(n_days, 1))
    t = z / np.sqrt(chi / df) * np.sqrt((df - 2) / df)      # unit-variance t shocks
    vol_mult = np.ones((n_days, 1))
    vol_mult[1200:1300] = 2.5                                # crisis regime
    rets = (t @ L.T) * vol_mult + ANN_RET / 252
    idx = pd.bdate_range(end="2026-11-09", periods=n_days)
    return pd.DataFrame(rets, index=idx, columns=ASSETS)


# ===== var_methods =====
def portfolio_returns(returns: pd.DataFrame, weights: pd.Series) -> pd.Series:
    return returns @ weights.reindex(returns.columns)


def historical_var(returns, weights, conf=0.99):
    pr = portfolio_returns(returns, weights).to_numpy()
    var = -np.percentile(pr, (1 - conf) * 100)
    tail = pr[pr <= -var]
    es = -tail.mean() if len(tail) else var
    return var, es


def parametric_var(returns, weights, conf=0.99):
    w = weights.reindex(returns.columns).to_numpy()
    mu = float(returns.mean().to_numpy() @ w)
    sigma = float(np.sqrt(w @ returns.cov().to_numpy() @ w))
    nd = NormalDist()
    z = nd.inv_cdf(1 - conf)                                  # negative number
    var = -(mu + z * sigma)
    es = -(mu - sigma * nd.pdf(z) / (1 - conf))
    return var, es


def monte_carlo_var(returns, weights, conf=0.99, n_sims=10_000, seed=0):
    w = weights.reindex(returns.columns).to_numpy()
    mu = returns.mean().to_numpy()
    cov = returns.cov().to_numpy()
    rng = np.random.default_rng(seed)
    L = np.linalg.cholesky(cov)
    sims = mu + rng.standard_normal((n_sims, len(mu))) @ L.T
    pr = sims @ w
    var = -np.percentile(pr, (1 - conf) * 100)
    tail = pr[pr <= -var]
    return var, -tail.mean()


METHODS = {
    "Historical": historical_var,
    "Parametric": parametric_var,
    "Monte Carlo": monte_carlo_var,
}


# ===== backtest =====
def kupiec_pof(exceptions: int, n_obs: int, conf: float):
    """Returns (LR statistic, p-value). H0: exception rate == 1 - conf. Chi-square(1 df)."""
    p = 1 - conf
    x, T = exceptions, n_obs
    phat = x / T
    def ll(prob):
        a = (T - x) * math.log(1 - prob) if prob < 1 else 0.0
        b = x * math.log(prob) if x > 0 else 0.0
        return a + b
    lr = -2 * (ll(p) - ll(phat)) if 0 < x < T else -2 * (ll(p) - (0.0))
    p_value = math.erfc(math.sqrt(max(lr, 0) / 2))            # chi2(1) survival function
    return lr, p_value


def rolling_var(returns, weights, method, conf=0.99, window=250, **kw):
    func = METHODS[method]
    out = []
    for t in range(window, len(returns)):
        hist = returns.iloc[t - window:t]
        out.append(func(hist, weights, conf, **kw)[0])
    return pd.Series(out, index=returns.index[window:], name=method)


def backtest(returns, weights, conf=0.99, window=250, mc_sims=3000):
    pr = portfolio_returns(returns, weights).iloc[window:]
    var_series, rows = {}, []
    for m in METHODS:
        kw = {"n_sims": mc_sims} if m == "Monte Carlo" else {}
        v = rolling_var(returns, weights, m, conf, window, **kw)
        var_series[m] = v
        exc = int((-pr > v).sum())
        lr, pval = kupiec_pof(exc, len(pr), conf)
        rows.append({
            "Method": m, "Obs": len(pr), "Exceptions": exc,
            "Expected": round(len(pr) * (1 - conf), 1),
            "Exception rate": f"{exc / len(pr):.2%}",
            "Kupiec LR": round(lr, 3), "p-value": round(pval, 4),
            "Result": "PASS" if pval > 0.05 else "REJECT",
        })
    return pd.DataFrame(rows), pd.DataFrame(var_series), pr


# ===== run =====
pd.set_option("display.width", 160)
os.makedirs("results", exist_ok=True)

returns = simulate_returns()
print(f"Data: {returns.shape[0]} days, {returns.shape[1]} assets | Portfolio value: {PORTFOLIO_VALUE:,.0f}\n")

# 1) Compare methods on the full sample, 95% and 99%
rows = []
for conf in (0.95, 0.99):
    for name, fn in METHODS.items():
        var, es = fn(returns.iloc[-500:], WEIGHTS, conf)   # last 500 days
        rows.append({"Confidence": f"{conf:.0%}", "Method": name,
                     "VaR %": f"{var:.2%}", "VaR $": f"{var * PORTFOLIO_VALUE:,.0f}",
                     "ES %": f"{es:.2%}", "ES $": f"{es * PORTFOLIO_VALUE:,.0f}"})
comparison = pd.DataFrame(rows)
print("=== 1-day VaR / ES comparison (last 500 days) ===")
print(comparison.to_string(index=False))
comparison.to_csv("results/var_comparison.csv", index=False)

# 2) Rolling backtest + Kupiec test at 99%
print("\n=== Backtest (99% VaR, 250-day window) with Kupiec POF test ===")
bt, var_series, realised = backtest(returns, WEIGHTS, conf=0.99, window=250)
print(bt.to_string(index=False))
bt.to_csv("results/kupiec_backtest.csv", index=False)

# 3) Plot
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(realised.index, realised, color="grey", lw=0.6, label="Portfolio daily return")
for m in var_series:
    ax.plot(var_series.index, -var_series[m], lw=1.2, label=f"-VaR 99% ({m})")
ax.set_title("Portfolio returns vs 99% VaR by method")
ax.legend(loc="lower left", fontsize=8)
fig.tight_layout()
fig.savefig("results/var_backtest.png", dpi=130)
print("\nSaved: results/var_comparison.csv, results/kupiec_backtest.csv, results/var_backtest.png")


Data: 2000 days, 7 assets | Portfolio value: 10,000,000

=== 1-day VaR / ES comparison (last 500 days) ===
Confidence      Method VaR %   VaR $  ES %    ES $
       95%  Historical 1.01% 101,080 1.51% 151,177
       95%  Parametric 1.04% 103,812 1.30% 130,419
       95% Monte Carlo 1.02% 101,732 1.29% 129,211
       99%  Historical 1.74% 174,364 2.47% 246,942
       99%  Parametric 1.47% 147,206 1.69% 168,783
       99% Monte Carlo 1.47% 147,452 1.69% 168,564

=== Backtest (99% VaR, 250-day window) with Kupiec POF test ===
     Method  Obs  Exceptions  Expected Exception rate  Kupiec LR  p-value Result
 Historical 1750          28      17.5          1.60%      5.384   0.0203 REJECT
 Parametric 1750          33      17.5          1.89%     11.003   0.0009 REJECT
Monte Carlo 1750          33      17.5          1.89%     11.003   0.0009 REJECT

Saved: results/var_comparison.csv, results/kupiec_backtest.csv, results/var_backtest.png
